# Configuración del Dataset (Carga y Rutas)
Este bloque realiza tres acciones clave:
1. Permite la carga manual del archivo `.zip`.
2. Descomprime el contenido.
3. Configura automáticamente `DATA_ROOT` detectando la carpeta real de los datos.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

# 1. CARGA DEL ARCHIVO
# Si el archivo no existe en el sistema, solicitar carga
archivo_zip = '/content/landslide4sense.zip'

if not os.path.exists(archivo_zip):
    print("Por favor, sube el archivo 'landslide4sense.zip' que comprimiste:")
    uploaded = files.upload()
else:
    print(f"✓ El archivo {archivo_zip} ya se encuentra en el sistema.")

# 2. DESCOMPRESIÓN
print("Descomprimiendo archivos... esto puede tardar un momento.")
with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
    # Extraemos en una carpeta limpia para evitar desorden
    zip_ref.extractall('/content/dataset_temp')

# 3. AJUSTE AUTOMÁTICO DE RUTAS (Basado en la estructura de tu imagen)
# Buscamos recursivamente dónde quedó 'TrainData'
detector_rutas = list(Path('/content/dataset_temp').glob('**/TrainData/img'))

if detector_rutas:
    # La raíz es dos niveles arriba de la carpeta 'img'
    # Siguiendo tu imagen: /content/dataset_temp/landslide4sense/landslide4sense/
    DATA_ROOT = detector_rutas[0].parent.parent
    
    print(f"\n✅ ¡CONFIGURACIÓN EXITOSA!")
    print(f"Ruta detectada: {DATA_ROOT}")
    
    # Definición de variables globales para los notebooks
    img_list = sorted(list((DATA_ROOT / "img").glob("*.h5")))
    mask_list = sorted(list((DATA_ROOT / "mask").glob("*.h5")))
    
    print(f"Muestras de entrenamiento encontradas: {len(img_list)}")
    
    # Verificar ValidData
    val_path = DATA_ROOT.parent / "ValidData" / "img"
    if val_path.exists():
        val_count = len(list(val_path.glob("*.h5")))
        print(f"Muestras de validación encontradas: {val_count}")
else:
    print("❌ ERROR: No se encontró la estructura 'TrainData/img' dentro del zip.")
    print("Contenido extraído:")
    !ls -R /content/dataset_temp | head -n 20